# 03. Первичный пространственный анализ

**ЛР 2, шаг 8.** Цель: получить первые ответы на вопросы заказчика и проверить, что выбранные данные и методы работают. Это ещё не продукт, а проверка гипотез, на основе которой уточняются бэклог и план.

Пример AquaMap отвечает на три вопроса:
1. Сколько точек питьевой воды в каждом районе и какова их плотность?
2. Какая доля жилых зданий находится в пределах 500 м от ближайшей точки?
3. Какие парки остаются без питьевой воды?

Замените вопросы и операции своими. Типовые операции: проекция в метры, буфер, пространственное объединение (spatial join), ближайший сосед, пересечение (overlay), агрегирование по районам, картограмма (choropleth).\n\n[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/03_spatial_analysis.ipynb)\n\nЧтобы кнопка открывала ваш ноутбук, замените в ссылке `OWNER/REPO` на владельца и имя своего репозитория (например, `ivan-p/pm-lab-00-aquamap`). Другие способы открыть ноутбук описаны в приложении Б.

## Настройка среды

Ноутбук рассчитан на **Google Colab** (как получить бесплатный доступ: приложение Б к заданию ЛР 2). Он работает и локально, в Jupyter или VS Code.

Что делает ячейка ниже:
1. В Colab ставит недостающие библиотеки и подключает Google Диск. Colab попросит разрешение на доступ к Диску: выберите тот же аккаунт Google, под которым открыт Colab.
2. Находит рабочую папку группы на Диске. Все данные и результаты сохраняются туда, поэтому после отключения сессии ничего не пропадёт.
3. Создаёт внутри неё ту же структуру папок, что и в репозитории: `data/raw`, `data/manual`, `data/processed`, `docs/img`, `docs/maps`.

**Перед первым запуском** впишите в `PROJECT_FOLDER` имя папки группы на Google Диске.

Запуск всего ноутбука: **Среда выполнения → Выполнить все** (Runtime → Run all).

In [ ]:
# === Настройка среды: запустите эту ячейку первой ===
import sys, subprocess
from pathlib import Path

# Имя папки группы в «Моём диске» Google. Обычно совпадает с именем репозитория.
PROJECT_FOLDER = "pm-lab-10-Charging-station"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'geopandas>=1.0', 'mapclassify', 'folium'], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    if not BASE.exists():
        raise FileNotFoundError(
            f"На Google Диске нет папки «{PROJECT_FOLDER}». Создайте её или добавьте ярлык "
            "общей папки группы в «Мой диск» (приложение Б, шаг 2) и запустите ячейку снова.")
else:
    BASE = Path("..")   # локальный запуск из папки notebooks/ репозитория

RAW, MANUAL, PROCESSED = BASE / "data" / "raw", BASE / "data" / "manual", BASE / "data" / "processed"
IMG, MAPS = BASE / "docs" / "img", BASE / "docs" / "maps"
for folder in (RAW, MANUAL, PROCESSED, IMG, MAPS):
    folder.mkdir(parents=True, exist_ok=True)
print("Среда:", "Google Colab" if IN_COLAB else "локальная", "· рабочая папка:", BASE.resolve())

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium

# Радиусы для анализа
RADIUS_M = 2000 # 2 км - радиус покрытия одной зарядки

boundary  = gpd.read_file(PROCESSED / "boundary.gpkg")
METRIC_CRS = boundary.estimate_utm_crs()

# Загружаем наши слои
districts = gpd.read_file(PROCESSED / "districts.gpkg").to_crs(METRIC_CRS)[["name", "geometry"]]
stations  = gpd.read_file(PROCESSED / "charging_stations.gpkg").to_crs(METRIC_CRS)
malls     = gpd.read_file(PROCESSED / "malls.gpkg").to_crs(METRIC_CRS)
parking   = gpd.read_file(PROCESSED / "parking.gpkg").to_crs(METRIC_CRS)

# Для удобства переведем полигоны ТЦ и парковок в точки
malls["geometry"] = malls.geometry.representative_point()
parking["geometry"] = parking.geometry.representative_point()

print(len(districts), "районов,", len(stations), "зарядок,", len(malls), "ТЦ,", len(parking), "парковок")

## Вопрос 1. Точки по районам и плотность

In [ ]:
districts["area_km2"] = districts.area / 1e6
zones = districts[["name", "geometry"]].rename(columns={"name": "district"})

# Считаем парковки по районам
pk_in = gpd.sjoin(parking, zones, predicate="within", how="inner")
districts["parking_count"] = districts["name"].map(pk_in.groupby("district").size()).fillna(0).astype(int)

# Считаем ТЦ по районам
ml_in = gpd.sjoin(malls, zones, predicate="within", how="inner")
districts["malls_count"] = districts["name"].map(ml_in.groupby("district").size()).fillna(0).astype(int)

# Считаем плотность спроса (парковок на 10 кв.км)
districts["parking_per_10km2"] = (districts["parking_count"] / districts["area_km2"] * 10).round(2)

districts.drop(columns="geometry")

### Дополнительно: точки на 10 тыс. жителей

Если в `data/manual/population_by_district.csv` есть численность населения по районам (столбцы `district`, `population`, `year`, `source`), ноутбук посчитает точки на 10 тыс. жителей. Названия районов в таблице должны совпадать с названиями в слое районов.

In [ ]:
POP_FILE = MANUAL / "population_by_district.csv"
pop = pd.read_csv(POP_FILE) if POP_FILE.exists() else pd.DataFrame()
if len(pop):
    districts = districts.merge(pop[["district", "population"]], left_on="name", right_on="district",
                                how="left").drop(columns="district")
    districts["points_per_10k"] = (districts["points"] / districts["population"] * 10_000).round(2)
    missing = districts.loc[districts["population"].isna(), "name"].tolist()
    if missing:
        print("Нет численности для районов:", missing, "— проверьте названия в CSV")
else:
    print("Таблица населения не заполнена, расчёт на 10 тыс. жителей пропущен")
districts.drop(columns="geometry")

## Вопрос 2. Жилые здания в пешей доступности

In [ ]:
# Создаем буфер 2 км вокруг зарядок
station_buffers = stations.copy()
station_buffers["geometry"] = stations.geometry.buffer(RADIUS_M)

# Проверяем, какие парковки попали в зону покрытия
parking_in_coverage = gpd.sjoin(parking, station_buffers, how="inner", predicate="intersects")

# Убираем дубли по индексу (чтобы одна парковка не считалась дважды, если попала в 2 буфера)
parking_in_coverage = parking_in_coverage[~parking_in_coverage.index.duplicated()]

total_parking = len(parking)
covered_parking = len(parking_in_coverage)
percent = (covered_parking / total_parking) * 100 if total_parking > 0 else 0

print(f"Всего парковок: {total_parking}")
print(f"В радиусе {RADIUS_M/1000} км от зарядок: {covered_parking}")
print(f"Доля покрытия: {percent:.1f}%")

# Добавляем информацию о покрытии в районы (для карты)
# Тут тоже убираем дубли по индексу перед группировкой
cov = gpd.sjoin(parking, zones, predicate="within", how="inner")
cov = cov[~cov.index.duplicated()]

# Проверяем, попала ли парковка в покрытие (по индексу)
cov["covered"] = cov.index.isin(parking_in_coverage.index)

district_cov = cov.groupby("district").agg(total=("covered", "size"), covered=("covered", "sum"))
district_cov["covered_pct"] = (district_cov["covered"] / district_cov["total"] * 100).round(1)

districts = districts.merge(district_cov[["covered_pct"]], left_on="name", right_index=True, how="left").fillna(0)

## Карты

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
districts.plot(ax=ax, color="whitesmoke", edgecolor="gray")

# Рисуем парковки
parking.plot(ax=ax, color="gray", alpha=0.3, markersize=2)
# Рисуем ТЦ
malls.plot(ax=ax, color="orange", markersize=15)
# Рисуем зону покрытия
station_buffers.plot(ax=ax, color="green", alpha=0.2)
# Рисуем зарядки
stations.plot(ax=ax, color="darkgreen", marker="*", markersize=150)

ax.set_title(f"Спрос (ТЦ, парковки) и зона {RADIUS_M/1000} км от зарядок (© OpenStreetMap)")
ax.set_axis_off()
fig.savefig(IMG / "coverage_by_district.png", dpi=150, bbox_inches="tight")

In [ ]:
m = districts.to_crs(4326).explore(color="gray", style_kwds={"fillOpacity": 0.1}, name="Районы")
parking.to_crs(4326).explore(m=m, color="gray", name="Парковки", tooltip=["name"])
malls.to_crs(4326).explore(m=m, color="orange", name="ТЦ", tooltip=["name"])
station_buffers.to_crs(4326).explore(m=m, color="green", name=f"Зона {RADIUS_M/1000} км")
stations.to_crs(4326).explore(m=m, color="darkgreen", name="Зарядки", tooltip=["operator", "amenity"])

folium.LayerControl().add_to(m)
m.save(MAPS / "analysis_v0.html")
m

## Выводы (заполните)

1. Короткий ответ на каждый вопрос (с цифрами).
2. Насколько можно доверять результату: какие ограничения данных влияют на выводы.
3. Что это значит для продукта: какие истории бэклога подтвердились, какие нужно добавить или убрать.
4. Какие риски из реестра изменились.

Карта `docs/img/coverage_by_district.png` вставляется в отчёт и в презентацию для заказчика.

Выводы:

Ответ на вопрос 1 (Спрос): Наибольшее количество парковок и ТЦ сосредоточено в центральных и густонаселенных районах (например, Ленинский, Октябрьский).

Ответ на вопрос 2 (Покрытие): В связи с тем, что в OSM всего 1 зарядная станция, зона покрытия в радиусе 2 км охватывает лишь малую долю парковок.

Ограничения данных: Данные OSM критически неполны в части зарядных станций (1 шт.), поэтому оценка текущего покрытия не отражает реальность. Однако данные о спросе (парковки, ТЦ) пригодны для анализа.

Для продукта: Бэклог подтверждается. Основной упор в модели надо делать на поиск точек притяжения, а не на покрытие существующих станций. Риск R1 (неполнота OSM) подтвержден.

## Перенос результатов в репозиторий

Карты из `docs/img` и `docs/maps` должны попасть в репозиторий через Pull Request. В Colab ячейка ниже соберёт их в архив `results_03.zip` и скачает на компьютер. Распакуйте архив и загрузите файлы в те же папки репозитория в своей ветке: **Add file → Upload files** (подробно в приложении Б, шаг 6).

Файлы больше 10 МБ в репозиторий не загружайте: ячейка о них предупредит. При локальной работе результаты уже лежат в папках репозитория, ячейку можно пропустить.

In [ ]:
import shutil
FOLDERS = [IMG, MAPS]
if IN_COLAB:
    from google.colab import files
    pack = Path("/content/results_03")
    shutil.rmtree(pack, ignore_errors=True)
    for folder in FOLDERS:
        for f in folder.rglob("*"):
            if f.is_file() and f.name != ".gitkeep":
                if f.stat().st_size > 10 * 1024 * 1024:
                    print(f"Пропущен (больше 10 МБ): {f.relative_to(BASE)}")
                    continue
                target = pack / f.relative_to(BASE)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, target)
    archive = shutil.make_archive(str(pack), "zip", pack)
    print("В архиве:", sorted(str(p.relative_to(pack)) for p in pack.rglob("*") if p.is_file()))
    files.download(archive)
else:
    print("Локальный запуск: результаты уже лежат в папках репозитория")